In [ ]:
from engine.utils import l2_normalise
from engine.rabitq import get_rabit, make_grid, get_rabit_grid
from engine.evp import get_evp_row_arg_partition_stable
from move import move_on_unit_sphere

import numpy as np

In [2]:
def get_next_normed_random_vec( d: int ) -> np.array:
    """
    :param d: the number of dimensions required
    :return: a l2 normed random vector of dimension d and
    """
    return l2_normalise(np.random.normal(0, 1, (d,)))


In [3]:
def angle_between_in_radians(v1: np.array, v2: np.array):
    """Returns the angle in radians between two vectors."""

    # 1. Calculate dot product and norms
    dot_prod = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # 2. Handle zero vectors (avoid division by zero)
    if norm_v1 == 0 or norm_v2 == 0:
        raise ValueError("Cannot calculate angle with a zero vector")

    # 3. Calculate cosine of the angle
    cos_theta = dot_prod / (norm_v1 * norm_v2)

    # 4. Clip values to [-1, 1] to handle floating-point errors
    # This prevents 'arccos' from returning NaN if value is slightly > 1
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    angle_rad = np.arccos(cos_theta)

    return angle_rad

In [4]:
def euc_distance( v1: np.array, v2: np.array) -> float:
    accum = 0
    for i in range(len(v1)):
        diff_squared = (v1[i] - v2[i])**2
        accum = accum + diff_squared
    return np.sqrt(accum)

In [5]:
# Test things are working correctly.

test_dim = 2
x = get_next_normed_random_vec(test_dim)
y = get_next_normed_random_vec(test_dim)

print(np.linalg.norm(x - y))
print(euc_distance(x, y))
print(angle_between_in_radians(x, y))


1.5893982051022735
1.5893982051022735
1.837023418004335


In [6]:
def get_avg_angle_in_dim(dimensions):
    angles = 0.0
    iters = 500

    for i in range(iters):
        a = get_next_normed_random_vec(dimensions)
        b = get_next_normed_random_vec(dimensions)
        angles += angle_between_in_radians(a, b)

    average_angle = angles / iters
    return average_angle

In [7]:
from move import move_on_unit_sphere

BOTH = 4
EVP = 3
RABIT = 2
NEITHER = 1

def move_and_test(old_vector: np.array, angle: float, dims: int, codebook) -> int:
    """
    :param vector: a l2 normed vector
    :param angle: the angle to move
    :return: One of BOTH, EVP, RABIT or NIETHER depending on what nodes have changed by the rotation.
    """
    old_evp = get_evp_row_arg_partition_stable(old_vector)  # or get_evp_row_arg_partition_stable or get_evp_ternary_row_mask
    old_rabit = get_rabit(old_vector)
    # old_rabit = get_rabit_grid(old_vector, codebook)
    evp_diff = False
    rabit_diff = False

    direction = get_next_normed_random_vec(dims)
    new_vector = move_on_unit_sphere(old_vector, direction, angle)  # Generate the moved vector

    evp = get_evp_row_arg_partition_stable(new_vector) # or get_evp_row_arg_partition_stable or get_evp_ternary_row_mask
    rabit = get_rabit(new_vector)
    # rabit = get_rabit_grid(new_vector,codebook)

    if not np.array_equal(old_evp, evp):
        evp_diff = True

    if not np.array_equal(old_rabit, rabit):
        rabit_diff = True

    if evp_diff and rabit_diff:
        return BOTH
    elif evp_diff:
        return EVP
    elif rabit_diff:
        return RABIT
    else:
        return NEITHER

In [8]:
from tqdm import tqdm
import pandas as pd
import os

In [9]:
iters = 1000
results=[]
output_path = "average_results.csv"

for dims in [3, 4, 5, 7, 10,100,200,400,600,800]:
    both_count = 0
    evp_count = 0
    rabit_count = 0
    neither_count = 0
    # average_angle = np.pi / 2 # 
    average_angle = get_avg_angle_in_dim(dims)
    # rabbit_codebook = make_grid(dims)
    for i in tqdm(range(iters)):
        vector = get_next_normed_random_vec(dims)
        result = move_and_test(vector, average_angle, dims, None)
        if result == EVP:
            evp_count += 1
        elif result == RABIT:
            rabit_count += 1
        elif result == NEITHER:
            neither_count += 1
        else:
            both_count += 1

    res = {"Dimensions": dims, "angle": average_angle, "both_count": both_count, "evp_count": evp_count, "rabit_count": rabit_count,"neither_count":neither_count}
    results.append(res)
    df = pd.DataFrame(results)
    df.to_csv(output_path, mode='w')

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [02:31<00:00,  6.59it/s]


In [10]:
print(f"Average distances for {dimensions} dimensions")
print(f"Iterations: {iters} Dimension: {dimensions} Angle: {average_angle}")
print(f"% changed through angle:")
print(f"Both {both_count * 100/iters}%")
print(f"EVP {evp_count * 100/iters}%")
print(f"RABIT {rabit_count * 100/iters}%")
print(f"NEITHER {neither_count * 100/iters}%")


NameError: name 'dimensions' is not defined